# Mini Project 1 — Analysis Notebook

**Your name:** Ashlesha Hadkar 
**Dataset:** Week 6 / `KCC Data /combined_kcc_3states_en.csv` (merged Maharashtra, Karnataka, Uttar Pradesh KCC exports; produced by `combine_translate_kcc.py`).
**Date:**  6/05/2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [4]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Original source:** data.gov.in
https://api.data.gov.in/resource/cef25fe2-9231-4128-8aec-2c948fedd43f

**Data accessed from:** State-level KCC CSV folders under `Week 6/KCC Data /` (Maharashtra, Karnataka, Uttar Pradesh), combined with `combine_translate_kcc.py`, which also adds `KccAns_en` for translated answers when translation has been run.

**Why this dataset**

Farmer support systems like KCC are designed to  bring agricultural expertise directly to rural communities, but design intent and ground-level reality are not always the same thing — and that gap is a core HCD question. This dataset is a lens for examining 
who gets helped and who doesn't, how underserved communities interact with government-built technology, and whether a public support system is actually reaching the people it was built for.

**Three analytical questions:**

1. Which types of queries dominate each month, and how does the mix change across the year?
2. Are queries tagged to the right QueryType? A mismatch audit.
3. What share of queries are for Agronomic / Crop Advisory support, how does the topic breakdown differ by state, and is demand growing?

**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [5]:
# Load combined KCC CSV (built by combine_translate_kcc.py → KCC Data /combined_kcc_3states_en.csv)
from pathlib import Path

filename = 'combined_kcc_3states_en.csv'
kcc_dir = Path('KCC Data ')
candidates = [
    kcc_dir / filename,
    Path('Week 6') / kcc_dir / filename,
    Path('..') / kcc_dir / filename,
    Path('../Week 6') / kcc_dir / filename,
    Path('../..') / Path('Week 6') / kcc_dir / filename,
]
# This code finds the path to the CSV file for the KCC CSVs.
csv_path = next((path for path in candidates if path.exists()), None)
if csv_path is None:
    searched = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find '{filename}'. Run from Week 6/: python3 combine_translate_kcc.py (or --resume). Checked:\n{searched}"
    )
# This code reads the CSV file for the KCC CSVs.
df = pd.read_csv(csv_path, encoding='utf-8-sig', low_memory=False)

# Drop Maharashtra 2025 rows so all three states cover the same 2022-2024 window
df = df[~((df['StateName'].astype(str).str.strip() == 'MAHARASHTRA') & (df['year'] == 2025))]
df = df.reset_index(drop=True)

df['QueryType_clean'] = df['QueryType'].astype(str).str.strip()

print(df.shape)
df.head()

(5400, 18)


,BlockName,Category,CreatedOn,Crop,DistrictName,KCCCallID,KccAns,QueryText,QueryType,Season,Sector,StateName,day,month,year,_source_file,KccAns_en,QueryType_clean
0,MADHA,Sugar and Starch Crops,2022-04-24T17:01:29.06,Sugarcane (Noble Cane),SOLAPUR,1197635,ऊस पिकासाठी-१२ ६१ ०० ६० ग्रॅम + मायक्रोला ३० म...,FARMER ASKED ABOUT FERTILIZER SPRAY ON SUGARCA...,Fertilizer Use and Availability,NaN,AGRICULTURE,MAHARASHTRA,24,4,2022,Maharashtra/2022/apr.csv,For sugarcane crop- 12 61 00 60 gm + micro wit...,Fertilizer Use and Availability
1,AUNDHA NAGNATH,Condiments and Spices,2022-04-24T17:02:24.237,Turmeric,HINGOLI,1197668,हवामान विभागाच्या अंदाजानुसार आपल्या औंढा-नागन...,Farmer asked query on Weather,Weather,NaN,HORTICULTURE,MAHARASHTRA,24,4,2022,Maharashtra/2022/apr.csv,According to the forecast of the Meteorologica...,Weather
2,JAMKHED,Others,2022-04-25T09:01:04.307,Others,AHMADNAGAR,1200438,⦁\tप्रधानमंत्री किसान सन्मान निधी योजना संबंधी...,Farmer asked about PM Kisan Sanman Nidhi Yojan...,Government Schemes,NaN,AGRICULTURE,MAHARASHTRA,25,4,2022,Maharashtra/2022/apr.csv,⦁ For more information regarding Prime Ministe...,Government Schemes
3,PAUNI,Others,2022-04-25T09:05:43.497,Others,BHANDARA,1200546,पी एम किसान साठी तुमच्या जवळील तहसील ऑफिस ला ज...,FARMER ASKED ABOUT PM KISAN ?,Government Schemes,NaN,AGRICULTURE,MAHARASHTRA,25,4,2022,Maharashtra/2022/apr.csv,For PM Kisan visit your nearest tehsil office ...,Government Schemes
4,MANGALVEDHE,Others,2022-04-25T09:16:57.56,Others,SOLAPUR,1200431,"हवामान विभागाच्या अंदाजानुसार,आपल्या तालुक्यात...",Farmer asked query on Weather,Weather,NaN,AGRICULTURE,MAHARASHTRA,25,4,2022,Maharashtra/2022/apr.csv,According to the forecast of Meteorological De...,Weather


In [6]:
# Check column types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5400 entries, 0 to 5399
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   BlockName        5400 non-null   object 
 1   Category         5400 non-null   object 
 2   CreatedOn        5400 non-null   object 
 3   Crop             5400 non-null   object 
 4   DistrictName     5400 non-null   object 
 5   KCCCallID        5400 non-null   int64  
 6   KccAns           5390 non-null   object 
 7   QueryText        5400 non-null   object 
 8   QueryType        5400 non-null   object 
 9   Season           0 non-null      float64
 10  Sector           5400 non-null   object 
 11  StateName        5400 non-null   object 
 12  day              5400 non-null   int64  
 13  month            5400 non-null   int64  
 14  year             5400 non-null   int64  
 15  _source_file     5400 non-null   object 
 16  KccAns_en        5390 non-null   object 
 17  QueryType_clea

In [7]:
# Summary statistics for numeric columns
df.describe()

,KCCCallID,Season,day,month,year
count,5.400000e+03,0.0,5400.000000,5400.000000,5400.000000
mean,3.125479e+06,NaN,14.819074,6.500000,2023.000000
std,2.079564e+06,NaN,8.347631,2.500232,0.816572
min,5.557060e+05,NaN,1.000000,3.000000,2022.000000
25%,1.337276e+06,NaN,8.000000,4.000000,2022.000000
50%,2.589199e+06,NaN,14.000000,6.500000,2023.000000
75%,4.927158e+06,NaN,23.000000,9.000000,2024.000000
max,7.139959e+06,NaN,31.000000,10.000000,2024.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1: Which types of queries dominate each month, and how does the mix change across the year?**

In [8]:
# Q1: Monthly query-type mix (all states combined, 100% stacked)
q1 = df.copy()
q1['QueryType_clean'] = q1['QueryType'].astype(str).str.strip()
q1['month'] = q1['month'].astype(int)

top_types = q1['QueryType_clean'].value_counts().head(5).index.tolist()
q1['QueryType_grouped'] = q1['QueryType_clean'].where(q1['QueryType_clean'].isin(top_types), 'Other')

monthly_mix = (
    q1.groupby(['month', 'QueryType_grouped'])
    .size()
    .reset_index(name='count')
)
monthly_mix['pct'] = 100 * monthly_mix['count'] / monthly_mix.groupby('month')['count'].transform('sum')

month_order = [3, 4, 6, 7, 9, 10]
month_labels = {3: 'Mar', 4: 'Apr', 6: 'Jun', 7: 'Jul', 9: 'Sep', 10: 'Oct'}
monthly_mix['month_label'] = monthly_mix['month'].map(month_labels)
monthly_mix['month_label'] = pd.Categorical(
    monthly_mix['month_label'],
    categories=[month_labels[m] for m in month_order],
    ordered=True,
)

print('Monthly query-type share (all states):')
display(monthly_mix.sort_values(['month', 'QueryType_grouped']))

heatmap_pivot = monthly_mix.pivot(index='QueryType_grouped', columns='month_label', values='pct').fillna(0)
month_col_order = [month_labels[m] for m in month_order]
heatmap_pivot = heatmap_pivot[month_col_order]

fig_q1 = px.imshow(
    heatmap_pivot,
    text_auto='.1f',
    aspect='auto',
    title='Weather and Government Schemes dominate monthly KCC query volume',
    labels={'x': 'Month', 'y': 'Query type', 'color': 'Share (%)'},
)
fig_q1.update_layout(xaxis_title='Month', yaxis_title='Query type')
fig_q1.show()

Monthly query-type share (all states):


,month,QueryType_grouped,count,pct,month_label
0,3,Government Schemes,251,27.888889,Mar
1,3,Market Information,64,7.111111,Mar
2,3,Nutrient Management,39,4.333333,Mar
3,3,Other,100,11.111111,Mar
4,3,Plant Protection,96,10.666667,Mar
5,3,Weather,350,38.888889,Mar
6,4,Government Schemes,213,23.666667,Apr
7,4,Market Information,82,9.111111,Apr
8,4,Nutrient Management,28,3.111111,Apr
9,4,Other,131,14.555556,Apr


**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Question 2: Are queries tagged to the right QueryType? A mismatch audit.**

In [9]:
# Q2: QueryType mismatch audit
import re

q2 = df[['QueryText', 'QueryType_clean', 'KccAns', 'Crop', 'StateName']].copy()
q2['query_lower'] = q2['QueryText'].astype(str).str.lower()

pest_pat = r'pest|insect|worm|caterpillar|fungus|fungal|disease|blight|rot|control.*attack'
spray_pat = r'\bspray\b'
scheme_pat = r'scheme|yojana|kisan|government|subsid|loan|credit|insurance'

pest_ok = {'Plant Protection', 'Disease Management', 'Weed Management',
           'Bio-Pesticides and Bio-Fertilizers'}
scheme_ok = {'Government Schemes', 'Crop Insurance', 'Credit'}

flags = []
for idx, row in q2.iterrows():
    txt = row['query_lower']
    qt = row['QueryType_clean']
    reason = None
    if re.search(pest_pat, txt) and qt not in pest_ok:
        reason = f'QueryText mentions pest/disease but tagged "{qt}"'
    elif re.search(scheme_pat, txt) and qt not in scheme_ok:
        reason = f'QueryText mentions scheme/gov but tagged "{qt}"'
    elif re.search(spray_pat, txt) and qt not in pest_ok and qt != 'Fertilizer Use and Availability':
        reason = f'QueryText mentions spray but tagged "{qt}"'
    if reason:
        flags.append({'row': idx, 'reason': reason, 'QueryText': str(row['QueryText'])[:120],
                      'TaggedAs': qt, 'State': row['StateName']})

flagged = pd.DataFrame(flags)
total_flagged = len(flagged)
print(f'Rows with a potential QueryType mismatch: {total_flagged} / {len(df)} ({100*total_flagged/len(df):.1f}%)\n')

summary = flagged.groupby('TaggedAs').size().reset_index(name='mismatch_count').sort_values('mismatch_count', ascending=False)
print('Mismatches by current QueryType tag:')
display(summary)

print('\nSample flagged rows:')
display(flagged.head(10))

fig_q2 = px.treemap(
    summary,
    path=['TaggedAs'],
    values='mismatch_count',
    title=f'QueryType tags with the most keyword mismatches ({total_flagged} total flagged rows)',
)
fig_q2.update_traces(textinfo='label+value')
fig_q2.show()


Rows with a potential QueryType mismatch: 59 / 5400 (1.1%)

Mismatches by current QueryType tag:


,TaggedAs,mismatch_count
9,Nutrient Management,22
11,Plant Protection,7
7,Government Schemes,5
2,Cultural Practices,4
16,Training and Exposure Visits,3
5,Fertilizer Use and Availability,3
12,"Post Harvest Management (Cleaning, Grading, Pa...",2
0,Agriculture Mechanization,2
8,Market Information,1
1,Cattle shed Planning and Management,1



Sample flagged rows:


,row,reason,QueryText,TaggedAs,State
0,104,"QueryText mentions scheme/gov but tagged ""Agri...",farmer asked about link of solar pump yojana ?,Agriculture Mechanization,MAHARASHTRA
1,156,"QueryText mentions pest/disease but tagged ""Po...",FARMER ASKED ABOUT WILT DISEASE ON TOMATO CROP?,"Post Harvest Management (Cleaning, Grading, Pa...",MAHARASHTRA
2,307,"QueryText mentions spray but tagged ""Nutrient ...",FARMER ASKED ABOUT FERTILIZER SPRAY FOR GROWTH...,Nutrient Management,MAHARASHTRA
3,343,"QueryText mentions spray but tagged ""Nutrient ...",Farmer asked about fertilizer spray on orange ?,Nutrient Management,MAHARASHTRA
4,390,"QueryText mentions spray but tagged ""Nutrient ...",FARMER ASKED ABOUT FERTILIZER SPRAY FOR GROWTH...,Nutrient Management,MAHARASHTRA
5,701,"QueryText mentions pest/disease but tagged ""Fe...",FARMER ASKED ABOUT CONTROL OF BLIGHT ATTACK IN...,Fertilizer Use and Availability,MAHARASHTRA
6,717,"QueryText mentions spray but tagged ""Nutrient ...",Fertilizer spray for sesame ?,Nutrient Management,MAHARASHTRA
7,771,"QueryText mentions spray but tagged ""Nutrient ...",ASKED ABOUT GROWTH SPRAY FOR CORIANDER CROP?\n,Nutrient Management,MAHARASHTRA
8,815,"QueryText mentions scheme/gov but tagged ""Orga...",Farmer asked about Government Scheme ?\n,Organic Farming,MAHARASHTRA
9,922,"QueryText mentions spray but tagged ""Nutrient ...",FARMER ASKED ABOUT FERTILIZER SPRAY FOR GROWTH...,Nutrient Management,MAHARASHTRA


**Interpretation:**  
*(What does this result tell you?)*

**Question 3: What share of queries are for Agronomic / Crop Advisory support, how does the topic breakdown differ by state, and is demand growing?**

In [10]:
# Q3: Agronomic / Crop Advisory analysis
AGRONOMIC_TYPES = {
    'Plant Protection', 'Fertilizer Use and Availability', 'Nutrient Management',
    'Weed Management', 'Cultural Practices', 'Water Management',
    'Seeds and Planting Material', 'Seeds', 'Disease Management',
    'Bio-Pesticides and Bio-Fertilizers', 'Field Preparation', 'Soil Testing',
    'Varieties', 'Sowing Time and Weather', 'Organic Farming', 'Soil Health Card',
    'Nursery Management', 'Vegetative Propagation and Tissue Culture',
    'Old/Senile Orchard Rejuvenation', 'Spices and Condiment Crops',
    'Water Management, Micro Irrigation',
}

q3 = df.copy()
q3['StateName'] = q3['StateName'].astype(str).str.strip()
q3['query_group'] = q3['QueryType_clean'].apply(
    lambda x: 'Agronomic / Crop Advisory' if x in AGRONOMIC_TYPES else 'Administrative / Informational'
)

# (a) Share by state
share = (
    q3.groupby(['StateName', 'query_group'])
    .size()
    .reset_index(name='count')
)
totals = share.groupby('StateName')['count'].transform('sum')
share['pct'] = 100 * share['count'] / totals

print('=== (a) Agronomic vs Administrative share by state ===')
display(share)

fig_a = px.bar(
    share,
    x='StateName',
    y='pct',
    color='query_group',
    barmode='stack',
    title='Agronomic / Crop Advisory vs Administrative queries by state',
    labels={'pct': '% of queries', 'StateName': 'State', 'query_group': 'Query group'},
    text=share['pct'].round(1).astype(str) + '%',
)
fig_a.update_traces(textposition='inside')
fig_a.update_layout(yaxis_title='% of queries', legend_title='Query group')
fig_a.show()

# (b) Top agronomic sub-topics per state
agro = q3[q3['query_group'] == 'Agronomic / Crop Advisory'].copy()
topic_state = (
    agro.groupby(['StateName', 'QueryType_clean'])
    .size()
    .reset_index(name='count')
    .sort_values(['StateName', 'count'], ascending=[True, False])
)
top_topics = topic_state.groupby('StateName').head(5).reset_index(drop=True)

print('\n=== (b) Top 5 agronomic sub-topics per state ===')
display(top_topics)

import plotly.graph_objects as go

all_top_types = top_topics['QueryType_clean'].unique().tolist()
fig_b = go.Figure()
for state in sorted(top_topics['StateName'].unique()):
    state_data = topic_state[topic_state['StateName'] == state]
    vals = [int(state_data.loc[state_data['QueryType_clean'] == t, 'count'].sum()) for t in all_top_types]
    fig_b.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=all_top_types + [all_top_types[0]],
        fill='toself',
        name=state,
    ))
fig_b.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    title='Agronomic sub-topic demand profile by state',
)
fig_b.show()

# (c) Year trend
agro_year = (
    agro.groupby(['year', 'StateName'])
    .size()
    .reset_index(name='count')
    .sort_values('year')
)

print('\n=== (c) Agronomic query count by year and state ===')
display(agro_year)

fig_c = px.line(
    agro_year,
    x='year',
    y='count',
    color='StateName',
    markers=True,
    title='Agronomic / Crop Advisory query volume by year',
    labels={'count': 'Number of queries', 'year': 'Year', 'StateName': 'State'},
)
fig_c.update_layout(xaxis=dict(dtick=1))
fig_c.show()


=== (a) Agronomic vs Administrative share by state ===


,StateName,query_group,count,pct
0,KARNATAKA,Administrative / Informational,1406,78.111111
1,KARNATAKA,Agronomic / Crop Advisory,394,21.888889
2,MAHARASHTRA,Administrative / Informational,1195,66.388889
3,MAHARASHTRA,Agronomic / Crop Advisory,605,33.611111
4,UTTAR PRADESH,Administrative / Informational,1215,67.500000
5,UTTAR PRADESH,Agronomic / Crop Advisory,585,32.500000



=== (b) Top 5 agronomic sub-topics per state ===


,StateName,QueryType_clean,count
0,KARNATAKA,Plant Protection,132
1,KARNATAKA,Nutrient Management,102
2,KARNATAKA,Cultural Practices,79
3,KARNATAKA,Water Management,19
4,KARNATAKA,Sowing Time and Weather,16
5,MAHARASHTRA,Plant Protection,254
6,MAHARASHTRA,Fertilizer Use and Availability,97
7,MAHARASHTRA,Nutrient Management,60
8,MAHARASHTRA,Weed Management,57
9,MAHARASHTRA,Cultural Practices,51



=== (c) Agronomic query count by year and state ===


,year,StateName,count
0,2022,KARNATAKA,134
1,2022,MAHARASHTRA,181
2,2022,UTTAR PRADESH,175
3,2023,KARNATAKA,126
4,2023,MAHARASHTRA,164
5,2023,UTTAR PRADESH,168
6,2024,KARNATAKA,134
7,2024,MAHARASHTRA,260
8,2024,UTTAR PRADESH,242


**Interpretation:**  
*(What does this result tell you?)*

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [8]:
# Section 4 visualization: Agronomic vs Administrative share by state (stacked %)
viz = (
    q3.groupby(['StateName', 'query_group'])
    .size()
    .reset_index(name='count')
)
state_totals = viz.groupby('StateName')['count'].transform('sum')
viz['pct'] = 100 * viz['count'] / state_totals

fig_viz = px.bar(
    viz,
    x='StateName',
    y='pct',
    color='query_group',
    title='Maharashtra farmers seek the most on-farm agronomic help among the three states',
    labels={'pct': '% of total queries', 'StateName': 'State', 'query_group': 'Query group'},
    text=viz['pct'].round(1).astype(str) + '%',
)
fig_viz.update_layout(
    yaxis_title='Percentage of queries',
    xaxis_title='State',
    legend_title='Query group',
    barmode='stack',
)
fig_viz.update_traces(textposition='inside')
fig_viz.show()


**Chart rationale:**  
*(Why this chart type? What should the reader take away?)*

### Interactive drill-down (horizontal): State > District > Query Type

The icicle chart below shows the hierarchy on a horizontal axis so labels are easier to read. You can interact in two ways:

- **Click-to-drill:** Click a state bar to zoom into its districts, then click a district to see its query-type breakdown. Click the top/root area to zoom back out.
- **Modebar controls:** Use the chart toolbar to pan, zoom in/out, and reset the view.

In [9]:
sb = df.copy()
sb['StateName'] = sb['StateName'].astype(str).str.strip()
sb['DistrictName'] = sb['DistrictName'].astype(str).str.strip()
sb['QueryType_clean'] = sb['QueryType'].astype(str).str.strip()

sb_counts = (
    sb.groupby(['StateName', 'DistrictName', 'QueryType_clean'])
    .size()
    .reset_index(name='count')
)

fig_sb = px.icicle(
    sb_counts,
    path=['StateName', 'DistrictName', 'QueryType_clean'],
    values='count',
    title='KCC queries: State > District > Query Type (horizontal drill-down)',
)
fig_sb.update_traces(textinfo='label+value')
fig_sb.update_layout(margin=dict(t=60, l=10, r=10, b=10), height=700, dragmode='pan')
fig_sb.show(
    config={
        'displayModeBar': True,
        'scrollZoom': True,
        'modeBarButtonsToAdd': ['pan2d', 'zoomIn2d', 'zoomOut2d', 'resetScale2d'],
    }
)

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.